# 24. Chap2-2 Baseline SegFormer Seed 반복

matched factorial train split으로 SegFormer-B0를 학습하고, 같은 모델을 `eval_matched`와 `eval_stress`에 모두 평가합니다.

이 노트북은 Chapter 2-2의 핵심 baseline입니다. 이후 노트북은 여기서 저장되는 seed별 `sample_metrics.csv`를 사용합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch2_2_utils.py").exists():
    matches = (
        list(Path.cwd().glob("Deeplearning/*/2-2장/ch2_2_utils.py"))
        + list(Path.cwd().glob("Deeplearning/*/2*/chap2_2/ch2_2_utils.py"))
        + list(Path.cwd().glob("**/ch2_2_utils.py"))
    )
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "2-2장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch2_2_utils import *

paths = find_ch2_2_paths()
set_korean_font()
set_seed(7)
paths

Chapter22Paths(chap2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), chapter2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-1장'), chapter1_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/1장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/data/synthetic_metal_matched'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs/manifests'))

## 24-1. 실행 설정

In [2]:
if not (paths.data_root / "metadata" / "samples.csv").exists():
    generate_matched_factorial_dataset(paths.data_root)
samples = load_ch2_2_samples(paths.data_root)
manifests = create_ch2_2_manifests(samples, paths.runs_root)

MODEL_SEEDS = [0, 1, 2]
EPOCHS = 30
BATCH_SIZE = 8
LR = 1e-3
MODEL_NAME = SEGFORMER_B0_MODEL_NAME
USE_PRETRAINED = True

print("train:", manifests["train"])
print("eval_matched:", manifests["eval_matched"])
print("eval_stress:", manifests["eval_stress"])

train: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\manifests\standard\train_manifest.csv
eval_matched: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\manifests\standard\eval_matched_manifest.csv
eval_stress: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\manifests\standard\eval_stress_manifest.csv


## 24-2. Seed별 학습과 matched/stress 평가

In [4]:
run_root = paths.runs_root / "baseline_seed_repeats"
run_root.mkdir(parents=True, exist_ok=True)

run_dirs = []
for seed in MODEL_SEEDS:
    run_dir = run_root / f"seed_{seed}"
    run_dirs.append(run_dir)

    print("training baseline seed:", seed)
    train_segformer_experiment(
        train_manifest=manifests["train"],
        eval_manifest=manifests["eval_matched"],
        run_dir=run_dir,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        seed=seed,
        augment=False,
        model_name=MODEL_NAME,
        use_pretrained=USE_PRETRAINED,
    )


    stress_dir = run_dir / "eval_stress"
    if not (stress_dir / "sample_metrics.csv").exists():
        print("evaluating stress split seed:", seed)
        evaluate_saved_model_on_manifest(run_dir, manifests["eval_stress"], stress_dir, batch_size=BATCH_SIZE)
    else:
        print("skip existing stress eval:", stress_dir)

training baseline seed: 0


c:\Users\준승\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 208/208 [00:00<00:00, 32216.22it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs mode

epoch 01 | train_loss=0.3661
epoch 02 | train_loss=0.1368
epoch 03 | train_loss=0.1230
epoch 04 | train_loss=0.1060
epoch 05 | train_loss=0.0837
epoch 06 | train_loss=0.0649
epoch 07 | train_loss=0.0512
epoch 08 | train_loss=0.0398
epoch 09 | train_loss=0.0333
epoch 10 | train_loss=0.0264
epoch 11 | train_loss=0.0222
epoch 12 | train_loss=0.0236
epoch 13 | train_loss=0.0258
epoch 14 | train_loss=0.0226
epoch 15 | train_loss=0.0175
epoch 16 | train_loss=0.0163
epoch 17 | train_loss=0.0144
epoch 18 | train_loss=0.0131
epoch 19 | train_loss=0.0141
epoch 20 | train_loss=0.0140
epoch 21 | train_loss=0.0140
epoch 22 | train_loss=0.0342
epoch 23 | train_loss=0.1416
epoch 24 | train_loss=0.0771
epoch 25 | train_loss=0.0531
epoch 26 | train_loss=0.0333
epoch 27 | train_loss=0.0297
epoch 28 | train_loss=0.0233
epoch 29 | train_loss=0.0209
epoch 30 | train_loss=0.0188
skip existing stress eval: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\baseline_seed_repeats\seed_0\eval_str

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 11345.10it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([6, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


epoch 01 | train_loss=0.3961
epoch 02 | train_loss=0.1419
epoch 03 | train_loss=0.1266
epoch 04 | train_loss=0.1152
epoch 05 | train_loss=0.0950
epoch 06 | train_loss=0.0773
epoch 07 | train_loss=0.0599
epoch 08 | train_loss=0.0510
epoch 09 | train_loss=0.0447
epoch 10 | train_loss=0.0392
epoch 11 | train_loss=0.0346
epoch 12 | train_loss=0.0291
epoch 13 | train_loss=0.0246
epoch 14 | train_loss=0.0221
epoch 15 | train_loss=0.0192
epoch 16 | train_loss=0.0191
epoch 17 | train_loss=0.0178
epoch 18 | train_loss=0.0224
epoch 19 | train_loss=0.0217
epoch 20 | train_loss=0.0184
epoch 21 | train_loss=0.0160
epoch 22 | train_loss=0.0138
epoch 23 | train_loss=0.0139
epoch 24 | train_loss=0.0136
epoch 25 | train_loss=0.0114
epoch 26 | train_loss=0.0144
epoch 27 | train_loss=0.0116
epoch 28 | train_loss=0.0113
epoch 29 | train_loss=0.0110
epoch 30 | train_loss=0.0134
skip existing stress eval: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\baseline_seed_repeats\seed_1\eval_str

Loading weights: 100%|██████████| 208/208 [00:00<00:00, 10897.97it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([6])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([6, 256, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


epoch 01 | train_loss=0.4038
epoch 02 | train_loss=0.1363
epoch 03 | train_loss=0.1210
epoch 04 | train_loss=0.0992
epoch 05 | train_loss=0.0787
epoch 06 | train_loss=0.0599
epoch 07 | train_loss=0.0457
epoch 08 | train_loss=0.0353
epoch 09 | train_loss=0.0320
epoch 10 | train_loss=0.0341
epoch 11 | train_loss=0.0307
epoch 12 | train_loss=0.0251
epoch 13 | train_loss=0.0202
epoch 14 | train_loss=0.0193
epoch 15 | train_loss=0.0192
epoch 16 | train_loss=0.0176
epoch 17 | train_loss=0.0148
epoch 18 | train_loss=0.0152
epoch 19 | train_loss=0.0159
epoch 20 | train_loss=0.1114
epoch 21 | train_loss=0.0571
epoch 22 | train_loss=0.0368
epoch 23 | train_loss=0.0261
epoch 24 | train_loss=0.0208
epoch 25 | train_loss=0.0188
epoch 26 | train_loss=0.0176
epoch 27 | train_loss=0.0169
epoch 28 | train_loss=0.0148
epoch 29 | train_loss=0.0150
epoch 30 | train_loss=0.0136
skip existing stress eval: C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\baseline_seed_repeats\seed_2\eval_str

## 24-3. Seed 결과 병합

In [5]:
matched_metrics = collect_sample_metrics(run_dirs, eval_name="eval_matched")
stress_metrics = collect_sample_metrics(run_dirs, eval_name="eval_stress")
matched_path = paths.runs_root / "baseline_seed_matched_sample_metrics.csv"
stress_path = paths.runs_root / "baseline_seed_stress_sample_metrics.csv"
matched_metrics.to_csv(matched_path, index=False, encoding="utf-8-sig")
stress_metrics.to_csv(stress_path, index=False, encoding="utf-8-sig")

summary = pd.DataFrame([
    {
        "eval_name": "eval_matched",
        "n_rows": len(matched_metrics),
        "n_seeds": matched_metrics["model_seed"].nunique(),
        "mean_target_dice": matched_metrics["target_dice"].mean(),
        "mean_target_fnr": matched_metrics["target_fnr"].mean(),
        "worst_combo_dice": matched_metrics.groupby(["color_group", "shape_group", "defect_type"])["target_dice"].mean().min(),
    },
    {
        "eval_name": "eval_stress",
        "n_rows": len(stress_metrics),
        "n_seeds": stress_metrics["model_seed"].nunique(),
        "mean_target_dice": stress_metrics["target_dice"].mean(),
        "mean_target_fnr": stress_metrics["target_fnr"].mean(),
        "worst_combo_dice": stress_metrics.groupby(["color_group", "shape_group", "defect_type"])["target_dice"].mean().min(),
    },
])
summary.to_csv(paths.runs_root / "baseline_seed_summary.csv", index=False, encoding="utf-8-sig")
display(summary)
print(matched_path)
print(stress_path)

,eval_name,n_rows,n_seeds,mean_target_dice,mean_target_fnr,worst_combo_dice
0,eval_matched,720,3,0.531328,0.501806,0.0
1,eval_stress,480,3,0.000000,1.000000,0.0


C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\baseline_seed_matched_sample_metrics.csv
C:\Users\준승\Desktop\2026-1\Study\Deeplearning\Vision 응용\2-2장\runs\baseline_seed_stress_sample_metrics.csv


## 24-4. 결론

In [6]:
print("결론: matched와 stress split의 seed 반복 추론 결과를 저장했습니다.")
print("next notebook: 25_Chap2_2_Factor별_일반화_분석.ipynb")

결론: matched와 stress split의 seed 반복 추론 결과를 저장했습니다.
next notebook: 25_Chap2_2_Factor별_일반화_분석.ipynb
